# Module 6 - Deploy the Databricks App

Builds the React frontend, syncs source to the workspace, creates (or reuses) the Databricks App, deploys it, and prints the URL.

This replaces the 4-command shell flow in `SETUP.md` with a single notebook run.

In [ ]:
%run ./_config


In [ ]:
%pip install -q databricks-sdk pyyaml
dbutils.library.restartPython()

## 1. Read upstream config + the live warehouse id

In [ ]:
import os, subprocess, shutil, time, yaml
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

def _cfg(key, fallback=""):
    rows = spark.sql(f"""
      SELECT value FROM {CONFIG_TABLE}
      WHERE key='{key}'
    """).collect()
    return rows[0].value if rows else fallback

warehouse_id = _cfg("warehouse_id")
if not warehouse_id:
    serverless = [wh for wh in w.warehouses.list() if getattr(wh, 'enable_serverless_compute', False)]
    warehouse_id = serverless[0].id if serverless else next(iter(w.warehouses.list())).id
agent_endpoint = AGENT_ENDPOINT
print(f"warehouse_id  = {warehouse_id}")
print(f"agent_endpoint = {agent_endpoint}")


## 2. Locate the app source on the local filesystem (repo `/app`)

In [ ]:
# When the notebook is run from a Repo, /Workspace/Repos/<user>/disa-genai-workshop/app exists
# When run as a one-off serverless job from a workspace folder, we need the local checkout.
candidates = [
    "/Workspace/Users/" + w.current_user.me().user_name + "/disa-genai-workshop/app",
    "/Workspace/Repos/" + w.current_user.me().user_name + "/disa-genai-workshop/app",
]
APP_SRC_LOCAL = None
for p in candidates:
    if os.path.isdir(p):
        APP_SRC_LOCAL = p
        break
if APP_SRC_LOCAL is None:
    raise RuntimeError(
        "App source not found in workspace. Run `databricks workspace import-dir app /Workspace/Users/<you>/disa-genai-workshop/app` first."
    )
print(f"App source: {APP_SRC_LOCAL}")

## 3. Build the React frontend (skipped if `dist/` already committed)

In [ ]:
dist_dir = os.path.join(APP_SRC_LOCAL, "dist")
if not os.path.isdir(dist_dir) or not os.listdir(dist_dir):
    print("dist/ missing or empty; running npm build (requires Node available on driver).")
    subprocess.run(["npm", "install"], cwd=APP_SRC_LOCAL, check=True)
    subprocess.run(["npm", "run", "build"], cwd=APP_SRC_LOCAL, check=True)
else:
    print(f"Found existing dist/ at {dist_dir}; skipping npm build.")

## 4. Patch app.yaml with the live warehouse id

Writes to a copy under /tmp so we don't dirty the repo.

In [ ]:
STAGED = "/tmp/disa_app_staged"
shutil.rmtree(STAGED, ignore_errors=True)
shutil.copytree(APP_SRC_LOCAL, STAGED)

yml_path = os.path.join(STAGED, "app.yaml")
with open(yml_path) as f:
    cfg = yaml.safe_load(f)
for r in cfg.get("resources", {}).get("sql_warehouse", []):
    r["id"] = warehouse_id
with open(yml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("Patched app.yaml warehouse id ->", warehouse_id)

## 5. Sync staged app to a workspace path (this is what the App engine reads)

In [ ]:
WS_APP = "/Workspace/Users/" + w.current_user.me().user_name + "/disa-genai-workshop-app"
subprocess.run(["databricks", "workspace", "import-dir", STAGED, WS_APP, "--overwrite"], check=True)
print("Synced to", WS_APP)

## 6. Create (or reuse) the App, then deploy

In [ ]:
# APP_NAME comes from _config

try:
    app = w.apps.get(name=APP_NAME)
    print(f"App {APP_NAME} already exists -> {app.url}")
except Exception:
    print(f"Creating app {APP_NAME} ...")
    subprocess.run(["databricks", "apps", "create", APP_NAME], check=False)
    # poll until ready
    for _ in range(20):
        try:
            app = w.apps.get(name=APP_NAME)
            if app.compute_status and str(app.compute_status.state) in ("RUNNING", "ACTIVE", "State.ACTIVE"):
                break
        except Exception:
            pass
        time.sleep(15)

deploy = subprocess.run(
    ["databricks", "apps", "deploy", APP_NAME, "--source-code-path", WS_APP],
    capture_output=True, text=True
)
print(deploy.stdout)
print(deploy.stderr)
deploy.check_returncode()


## 7. Print the App URL

In [ ]:
app = w.apps.get(name=APP_NAME)
print("URL:", app.url)